In [1]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [ ]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

@dataclass
class HERCParams:
  quantile:float|None=None
  use_lw_shrinkage:bool
  k_max:int
  n_sims:int


In [94]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark


  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [33]:
class Optimizer:
  def __init__(self, optim_params, debug: bool = False):
    self.p = optim_params
    self.debug = debug

  def get_asset_w(self, returns: pd.DataFrame):
    T, N = returns.shape
    R = returns.values

    if N == 1:
      col = returns.columns[0]
      r = R.flatten()

      var = np.quantile(-r, self.p.quantile)
      tail = -r[-r >= var]

      cvar = tail.mean() if len(tail) else var
      cvar = max(cvar, 1e-6)

      return pd.Series([1.0], index=[col]), cvar

    y = cp.Variable(N, nonneg=True, name="raw_weights")
    u = cp.Variable(T, nonneg=True, name="slack_losses")
    b = np.ones(N) / N
    zeta = cp.Variable(name="VaR")

    CVaR = zeta + (1 / ((1 - self.p.quantile) * T)) * cp.sum(u)
    log_barrier = cp.sum(cp.multiply(b, cp.log(y)))

    eps = 1e-3
    reg = eps * cp.sum(y)

    constraints = [
        u >= -R @ y - zeta,
        u >= 0,
        y >= 1e-4,
        y <= 1e+3
    ]

    obj_fn = cp.Minimize(CVaR + reg - log_barrier)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL, verbose=self.debug)

    if problem.status not in ("optimal", "optimal_inaccurate"):
      raise RuntimeError(f"Solver failed for cluster {list(returns.columns)}: {problem.status}")

    w_raw = y.value
    w = w_raw / w_raw.sum()
    w = pd.Series(w, index=returns.columns)

    cvar_c = CVaR.value / w_raw.sum()

    return w, cvar_c

  def get_cluster_w(self, risk_measure: np.ndarray, eps: float = 1e-6):
    risk_measure = np.maximum(np.asarray(risk_measure), eps)
    inv_risk = 1 / risk_measure

    return inv_risk / inv_risk.sum()

In [34]:
class HERCOptimizer(Optimizer):
  def __init__(self, optim_params: HERCParams, debug=False, **kwargs):
    super().__init__(optim_params=optim_params, debug=debug, **kwargs)

  def _compute_cl_dispersion(self, dist_mtx, cluster_labels):
    unique_clusters = np.unique(cluster_labels)
    W_k = 0.0
    for c_id in unique_clusters:
      cluster_indices = np.where(cluster_labels == c_id)[0]
      cluster_dist = dist_mtx[np.ix_(cluster_indices, cluster_indices)]

      norm = 2.0 * len(cluster_indices)

      D_r = np.sum(cluster_dist ** 2)

      W_k += D_r / norm

    return W_k

  def _generate_null_reference(self, returns_df):
    N_samples, N_assets = returns_df.shape

    min_bounds = returns_df.min(axis=0).values
    max_bounds = returns_df.max(axis=0).values

    null_dists, null_linkages = [], []

    for _ in range(self.p.n_sims):
      null_returns = np.random.uniform(low=min_bounds, high=max_bounds, size=(N_samples, N_assets))
      null_corr = np.clip(np.corrcoef(null_returns, rowvar=False), -1.0, 1.0)
      null_dist = np.sqrt(2.0 * (1.0 - null_corr))

      condensed_dist = squareform(null_dist, checks=False)

      Z_null = linkage(condensed_dist, method="single")

      null_dists.append(null_dist)
      null_linkages.append(Z_null)
    return null_dists, null_linkages

  def _compute_ref_log_disp(self, null_dists, null_linkages, k):
    B = self.p.n_sims
    W_k_log = []

    for b in range(B):
      clusters = fcluster(null_linkages[b], t=k, criterion="maxclust")

      W_k = self._compute_cl_dispersion(null_dists[b], clusters)
      W_k_log.append(np.log(max(W_k, 1e-300)))

    W_k_log = np.array(W_k_log)
    E_W_k = np.mean(W_k_log)

    sdk = np.std(W_k_log, ddof=1) if B > 1 else 0.0
    s_k = sdk * np.sqrt(1.0 + 1.0 / B)

    return E_W_k, s_k

  def _get_k_clusters(self, dist_matrix, Z, returns_df):
    k_max = min(self.p.k_max, dist_matrix.shape[0] - 1)
    Gap_k, s_k_list = [], []
    null_dists, null_linkages = self._generate_null_reference(returns_df)
    k_range = list(range(1, k_max + 1))

    for k in k_range:
      clusters = fcluster(Z, t=k, criterion="maxclust")

      W_k_real = self._compute_cl_dispersion(dist_matrix, clusters)
      log_disp_k = np.log(max(W_k_real, 1e-300))

      E_W_k, s_k = self._compute_ref_log_disp(null_dists, null_linkages, k)

      Gap_k.append(E_W_k - log_disp_k)
      s_k_list.append(s_k)

    gaps = np.array(Gap_k)
    s_k_list = np.array(s_k_list)

    optimal_k = k_range[np.argmax(gaps)]
    for idx in range(len(k_range) - 1):
      if gaps[idx] >= gaps[idx + 1] - s_k_list[idx + 1]:
        optimal_k = k_range[idx]
        break

    optimal_k = max(optimal_k, 2) if dist_matrix.shape[0] > 1 else optimal_k
    return int(optimal_k)

  def get_clusters(self, returns: pd.DataFrame) -> pd.DataFrame:
    corr = returns.corr().values
    dist_mtx = np.sqrt(2.0 * (1.0 - corr))

    comp_disp = squareform(dist_mtx, checks=False)
    Z = linkage(comp_disp, method="single")

    optimal_k = self._get_k_clusters(dist_mtx, Z, returns)
    labels = fcluster(Z, t=optimal_k, criterion="maxclust")

    return pd.DataFrame({'Asset': returns.columns, 'Cluster': labels}), Z

  def _ordered_cluster_ids(self, Z, clusters_df):
    label_by_asset = clusters_df.set_index('Asset')['Cluster']
    leaf_order = leaves_list(Z)

    asset_order = clusters_df['Asset'].values[leaf_order]
    ordered_labels = label_by_asset.loc[asset_order].values

    seen, ordered_ids = set(), []
    for lbl in ordered_labels:
      if lbl not in seen:
        seen.add(lbl)
        ordered_ids.append(lbl)

    return ordered_ids

  def _recursive_bisect(self, ordered_ids, cluster_return_series):
    weights = {cid: 1.0 for cid in ordered_ids}

    def risk_of(ids):
      if len(ids) == 1:
        return self.get_asset_w.__self__ and None

      combo = np.mean([cluster_return_series[c] for c in ids], axis=0)

      var = np.quantile(-combo, self.p.quantile)
      tail = -combo[-combo >= var]
      cvar = tail.mean() if len(tail) else var
      return max(cvar, 1e-6)

    def node_risk(ids):
      if len(ids) == 1:
        r = cluster_return_series[ids[0]]
        var = np.quantile(-r, self.p.quantile)
        tail = -r[-r >= var]
        cvar = tail.mean() if len(tail) else var

        return max(cvar, 1e-6)

      combo = np.mean([cluster_return_series[c] for c in ids], axis=0)
      var = np.quantile(-combo, self.p.quantile)
      tail = -combo[-combo >= var]
      cvar = tail.mean() if len(tail) else var

      return max(cvar, 1e-6)

    def bisect(ids):
      if len(ids) <= 1:
        return

      mid = len(ids) // 2
      left, right = ids[:mid], ids[mid:]
      r_l, r_r = node_risk(left), node_risk(right)
      alpha = 1 - r_l / (r_l + r_r)

      for c in left:  weights[c] *= alpha
      for c in right: weights[c] *= (1 - alpha)

      bisect(left)
      bisect(right)

    bisect(ordered_ids)
    return weights

  def optimize_w(self, returns: pd.DataFrame):
    clusters_df, Z = self.get_clusters(returns)
    unique = np.unique(clusters_df.Cluster.values)

    w_by_cluster = {}
    cluster_return_series = {}

    for c_id in unique:
      cols_i = clusters_df.loc[clusters_df.Cluster == c_id, 'Asset'].values
      returns_i = returns[cols_i]

      w_i, cvar_i = self.get_asset_w(returns_i)
      w_by_cluster[c_id] = w_i

      cluster_return_series[c_id] = (returns_i.values @ w_i.values)

    ordered_ids = self._ordered_cluster_ids(Z, clusters_df)
    cluster_weights = self._recursive_bisect(ordered_ids, cluster_return_series)

    w_final = []
    for c_id, w_i in w_by_cluster.items():
      w_final.append(w_i * cluster_weights[c_id])

    w_df = pd.concat(w_final)
    return w_df

In [35]:
class Portfolio(DataStore, HERCOptimizer):
  def __init__(self, optim_params, debug: bool = False, **kwargs):
    super().__init__(debug=debug, optim_params=optim_params, **kwargs)

  def get_data(self, universe, start, end):
    data, benchmark = self._get_data(universe=universe, start=start, end=end)
    returns = data.pct_change().dropna()
    return returns, benchmark

  def optimize(self, returns, plot=True):
    w = self.optimize_w(returns)
    if plot:
      w.plot.bar()
      plt.show()
    return w

In [36]:
optim_params = HERCParams()

In [38]:
ptf = Portfolio(optim_params=optim_params, debug=False)
returns, benchmark = ptf.get_data(
    universe=test_universe,
    start="2021-01-01",
    end="2027-01-01"
)

/tmp/ipykernel_2097/473624180.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers_clean, start, end, interval)["Close"]
[*********************100%***********************]  14 of 14 completed


In [ ]:
from arch import arch_model
from scipy.stats import genpareto
from statsmodels.distributions.copula.api import StudentTCopula

In [108]:
@dataclass
class SimParams:
  quantile:float=0.95
  n_paths:int = 10000
  max_p:int = 5
  max_q:int = 5


In [109]:
class GARCH_EVT_COPULA:
  def __init__(self, sim_params, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )

    self.sim_params = sim_params
    self.debug = debug
    self.copula = None
    self.best_models = None
    self.best_params = None

  def fit_model(self, r):
    best_bic = np.inf
    best_p, best_q = 0, 1
    best_fit = None

    for p in range(self.sim_params.max_p + 1):
      for q in range(1, self.sim_params.max_p + 1):
        try:
          model = arch_model(
            r,
            mean='AR',
            lags=p,
            vol='GARCH',
            p=1,
            q=q,
            dist='t'
          )
          fit = model.fit(disp='off', show_warning=False)

          if fit.bic < best_bic:
              best_bic = fit.bic
              best_p = p
              best_q = q
              best_fit = fit
        except Exception:
          continue

      return best_p, best_q, best_bic, best_fit

  def fit_ar_garch(self, returns):
    filtered_resid = pd.DataFrame(index=returns.index, columns=returns.columns)
    self.best_models = {}
    self.best_params = {}

    for ticker in returns.columns:
      r = returns[ticker].dropna()
      if self.debug: print(f"Ticker: {ticker} | p: {p}")

      best_p, best_q, best_bic, best_fit = self.fit_model(r)

      mean = best_fit.params['nu']
      z_resid = (best_fit.resid - mean) / best_fit.conditional_volatility

      filtered_resid[ticker] = z_resid
      self.best_models[ticker] = best_fit
      self.best_params[ticker] = (best_p, best_q)

    return filtered_resid

  def get_pseudo_observations(self, residuals):
    uniform_resid = pd.DataFrame(index=residuals.index, columns=residuals.columns)

    self.margin_params = {}

    for ticker in residuals.columns:
      r = residuals[ticker].dropna().to_numpy()
      n = len(r)

      q_high = self.sim_params.quantile
      q_low = 1.0 - self.sim_params.quantile
      u_upper = np.percentile(r, q_high * 100)
      u_lower = np.percentile(r, q_low * 100)

      upper_tail = r[r > u_upper]
      lower_tail = r[r < u_lower]

      c_U, _, scale_U = genpareto.fit(upper_tail - u_upper, floc=0)
      c_L, _, scale_L = genpareto.fit(u_lower - lower_tail, floc=0)

      p_l = len(lower_tail) / n
      p_u = len(upper_tail) / n

      u_transformed = np.zeros_like(r, dtype=float)

      low_mask = r < u_lower
      if np.any(low_mask):
          u_transformed[low_mask] = p_l * (1.0 - genpareto.cdf(u_lower - r[low_mask], c_L, loc=0, scale=scale_L))

      high_mask = r > u_upper
      if np.any(high_mask):
          u_transformed[high_mask] = (1.0 - p_u) + p_u * genpareto.cdf(r[high_mask] - u_upper, c_U, loc=0, scale=scale_U)

      body_mask = (~low_mask) & (~high_mask)
      if np.any(body_mask):
          sorted_r = np.sort(r)
          emp_prob = np.searchsorted(sorted_r, r[body_mask]) / (n + 1)
          u_transformed[body_mask] = emp_prob

      uniform_resid[ticker] = u_transformed

      self.margin_params[ticker] = {
          'u_upper': u_upper, 'u_lower': u_lower,
          'c_U': c_U, 'scale_U': scale_U,
          'c_L': c_L, 'scale_L': scale_L,
          'p_l': p_l, 'p_u': p_u,
          'body_data': r[(r >= u_lower) & (r <= u_upper)]
      }

    return uniform_resid

  def inverse_semi_parametric_cdf(self, uniform_samples):
    tickers = list(self.margin_params.keys())
    simulated_residuals = pd.DataFrame(index=range(len(uniform_samples)), columns=tickers)

    for i, ticker in enumerate(tickers):
      u_col = uniform_samples[:, i] if isinstance(uniform_samples, np.ndarray)
      params = self.margin_params[ticker]

      p_l = params["p_l"]
      p_u = params["p_u"]

      real_col = np.zeros_like(u_col)

      low_mask = u_col < p_l
      if np.any(low_mask):
        real_col[low_mask] = (
            params["u_lower"] \
            - genpareto.ppf(
                1.0 - u_col[low_mask] / p_l,
                params["c_L"],
                loc=0,
                scale=params["scale_L"]
            )
        )

      high_mask = u_col > (1.0 - p_u)

      if np.any(high_mask):
        real_col[high_mask] = (
            params["u_upper"] \
            + genpareto.ppf(
                (u_col[high_mask] - (1.0 - p_u)) / p_u,
                params["c_U"],
                loc=0,
                scale=params["scale_U"]
            )
        )

      body_mask = (~low_mask) & (~high_mask)
      if np.any(body_mask):
        body_percentiles = (u_col[body_mask] - p_l) / (1.0 - p_u - p_l) * 100
        body_percentiles = np.clip(body_percentiles, 0, 100)
        real_col[body_mask] = np.percentile(params['body_data'], body_percentiles)

      simulated_residuals[ticker] = real_col

    return simulated_residuals



  def fit_copula_evt(self, residuals):
    uniform_resid = self.get_pseudo_observations(residuals)

    self.copula = StudentTCopula(dim=len(residuals.columns))
    self.copula.fit(uniform_resid)

    if self.debug:
      print(self.copula.summary())

    degrees_of_freedom = self.copula.df
    corr_mtx = self.copula.corr

    dim = len(residuals.columns)
    self.copula_params = {"df": degrees_of_freedom, "corr_mtx": corr_mtx}

  def fit(self, returns):
    residuals = self.fit_ar_garch(returns)
    self.fit_copula_evt(residuals)

  def generate_sample(self):
    simulated_uniforms = self.copula.rvs(self.sim_params.n_paths)

    simulated_residuals = self.inverse_semi_parametric_cdf(simulated_uniforms)

    simulated_returns = pd.DataFrame(
        index=simulated_residuals.index,
        columns=simulated_residuals.columns
    )


    for ticker in simulated_residuals.columns:
      params = self.best_models[ticker].params
      p, q = self.best_params[ticker]

      # pass simulated residuals back into the AR-GARCH


    return simulated_returns







  def sim_test(self, returns, w):
    T, N = returns.shape
    R = returns.values